# SB System - Mac CSV Import to PostgreSQL

Use this notebook on the MacBook after copying MT5 CSV exports from Windows into `data/raw/mt5_export`.

This notebook starts/uses local Docker PostgreSQL, imports the exported candle files, and verifies the stored data. It does **not** connect to MetaTrader 5.

## 1. Load Project Configuration

In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sb_system.market_data import (
    check_connection,
    create_db_engine,
    create_schema,
    fetch_candle_summary,
    load_config,
    upsert_candles,
)

config = load_config(PROJECT_ROOT / ".env")
input_dir = PROJECT_ROOT / "data" / "raw" / "mt5_export"

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"Input folder: {input_dir}")
print(f"Database URL configured: {bool(config.database_url)}")

## 2. Start Local PostgreSQL

This uses `docker-compose.yml`. Docker Desktop must be running on the MacBook.

In [ ]:
result = subprocess.run(
    ["docker", "compose", "up", "-d", "postgres"],
    cwd=PROJECT_ROOT,
)
if result.returncode != 0:
    raise RuntimeError("Failed to start local PostgreSQL. Check Docker Desktop is running.")

subprocess.run(["docker", "compose", "ps"], cwd=PROJECT_ROOT)

## 3. Connect and Create Schema

In [ ]:
engine = create_db_engine(config.database_url)
create_schema(engine)
check_connection(engine)

## 4. Inspect Export Folder

In [ ]:
if not input_dir.exists():
    raise FileNotFoundError(
        f"Missing {input_dir}. Copy data/raw/mt5_export from Windows into this project first."
    )

manifest_path = input_dir / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print(f"Manifest total rows: {manifest.get('total_rows')}")
    display(pd.DataFrame(manifest.get("files", [])))
else:
    print("No manifest.json found. Import will scan CSV files directly.")

csv_files = sorted(
    path for path in input_dir.rglob("*")
    if path.is_file() and (path.name.endswith(".csv") or path.name.endswith(".csv.gz"))
)
print(f"CSV files found: {len(csv_files)}")
csv_files[:10]

## 5. Import Candles into PostgreSQL

In [ ]:
total_rows = 0
for file_path in csv_files:
    candles = pd.read_csv(file_path)
    rows = upsert_candles(engine, candles)
    total_rows += rows
    print(f"{rows:8} candles <- {file_path}")

print(f"Total imported rows: {total_rows}")

## 6. Verify Candle Summary

In [ ]:
summary = fetch_candle_summary(engine)
summary

## 7. Preview Recent Candles

In [ ]:
with engine.connect() as conn:
    recent = pd.read_sql_query(
        text(
            """
            SELECT
                s.broker_symbol,
                c.timeframe,
                c.candle_time,
                c.open,
                c.high,
                c.low,
                c.close,
                c.tick_volume,
                c.spread
            FROM market.candles c
            JOIN market.symbols s ON s.symbol_id = c.symbol_id
            ORDER BY c.candle_time DESC
            LIMIT 50
            """
        ),
        conn,
    )

recent